# 02 · Encoding: one-hot + padding
Step 2 of the thesis workflow (after `01_jets_meta`).

**Input:** `data/raw/journal_entries_4203.csv` (never modified), `data/derived/meta.parquet` (notebook 01)
**Output:** `data/derived/X_onehot.npz` (sparse, rows in `meta.parquet` order) + `data/derived/onehot_features.json` (column names, feature groups, fitted parameters)

Decisions used here (details and rationale in `results/feature_engineering_plan.md`):

| ID | What | Setting |
|---|---|---|
| D9 | line order inside the padding | debit first, then account number, then amount (largest first) |
| D10 | scalers | fitted on FY1 (2023) only; FY2 values are not clipped |
| D4 | calendar features | `promptly`, `weekend`, `nwh` kept, continuous `lag_days` added |
| D11 | extra time features | `nwh_dist`, hour sin/cos, weekday sin/cos built as switchable groups, **off** in the main setting |

All candidate features are stored once. A model run selects columns by group with `select_columns()`, so no rebuild is needed to switch a group on or off.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    # walk up from the working directory until `marker` exists (same as notebook 01)
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from src.load import load_lines
from src import encode as fe

DATA = ROOT / RAW_REL
DERIVED = ROOT / "data" / "derived"
FY1 = 2023          # tuning year (D6); FY2 = 2024 is the reporting year
print("project root:", ROOT)

## 1. Load raw lines and the meta table

In [ ]:
lines = load_lines(DATA)                   # src/load.py (adds fy, line_no)
meta = pd.read_parquet(DERIVED / "meta.parquet")

print(lines.shape, meta.shape)
assert len(meta) == 102_429 and lines["posting_id"].nunique() == len(meta)
assert set(lines["posting_id"]) == set(meta["posting_id"])

## 2. Which fields are per line and which are per entry?

If a field has one value inside every `posting_id`, it is encoded **once per entry**, not once per line slot. Repeating it in 4 slots would count it up to 4 times in every distance (LOF, kNN) without adding information.

In [ ]:
g = lines.groupby("posting_id", sort=False)
check = {c: int((g[c].nunique() > 1).sum())
         for c in ["user", "tax_rate", "entered_date", "document_date", "promptly", "weekend", "nwh",
                   "gl_account", "cd_flag", "amount"]}
pd.Series(check, name="entries with >1 value")

In [ ]:
# expected: 0 for all entry-level fields; > 0 for gl_account, cd_flag, amount (line level)
for c in fe.ENTRY_COLS:
    assert check[c] == 0, c

### Line patterns (debit/credit sequence as written in the CSV)

In [ ]:
pattern = g["cd_flag"].agg("".join).rename("csv_pattern")
pat = (pd.DataFrame({"csv_pattern": pattern})
       .join(meta.set_index("posting_id")[["label"]])
       .groupby("csv_pattern")["label"].agg(entries="size", anomalies="sum"))
pat

In [ ]:
# the only 4-line entry: it alone fills slot 4, so the one-hot encoding will make it very isolated
four = meta.loc[meta["n_lines"] == 4, "posting_id"].tolist()
print(four)
lines[lines["posting_id"].isin(four)].drop(columns=["gl_account_name", "text"])

## 3. Build the encoding

In [ ]:
X, feature_info, params, sorted_lines, ent = fe.build_onehot(lines, meta, fy1=FY1)
print(X.shape, f"non-zeros: {X.nnz:,}")
pd.Series([f["group"] for f in feature_info]).value_counts().rename("columns per group")

## 4. D9 — does the line order matter in this dataset?

Count entries whose CSV line order differs from the D9 order. Each of these would get a different vector than an entry with the same content written in another order.

In [ ]:
changed = fe.order_changes(sorted_lines)
print(f"entries with CSV order != D9 order: {changed.sum():,} of {len(changed):,} "
      f"({changed.mean():.1%})")
(pd.DataFrame({"changed": changed}).join(pattern)
   .groupby("csv_pattern")["changed"].agg(entries="size", changed="sum"))

In [ ]:
# example for the thesis: one entry before and after sorting
ex = changed[changed].index[0]
cols = ["posting_id", "csv_pos", "slot", "cd_flag", "gl_account", "gl_account_name", "amount"]
sorted_lines.loc[sorted_lines["posting_id"] == ex, cols].sort_values("csv_pos")

In [ ]:
# tie-break check: same side and same account twice inside one entry?
dup = sorted_lines.groupby(["posting_id", "cd_flag", "gl_account"]).size()
print("entries needing the amount tie-break:", int((dup > 1).sum()))

## 5. Sanity checks

In [ ]:
names = [f["name"] for f in feature_info]
col = {n: i for i, n in enumerate(names)}
is1 = (meta["fy"] == FY1).to_numpy()

def colv(name):
    return np.asarray(X[:, col[name]].todense()).ravel()

# (a) every slot has exactly one account category and one side category (PAD included)
for s in range(1, fe.MAX_LINES + 1):
    a = [i for i, f in enumerate(feature_info) if f["slot"] == s and f["name"].startswith(f"s{s}_acct_")]
    c = [i for i, f in enumerate(feature_info) if f["slot"] == s and f["name"].startswith(f"s{s}_cd_")]
    assert (np.asarray(X[:, a].sum(axis=1)).ravel() == 1).all()
    assert (np.asarray(X[:, c].sum(axis=1)).ravel() == 1).all()

# (b) slot use matches n_lines
for s in range(1, fe.MAX_LINES + 1):
    filled = colv(f"s{s}_acct_PAD") == 0
    assert (filled == (meta["n_lines"].to_numpy() >= s)).all()

# (c) scaled values: FY1 in [0, 1], FY2 may leave the range (not clipped, D10)
for nm in ["s1_amount", "lag_days", "nwh_dist"]:
    v = colv(nm)
    print(f"{nm:10s} FY1 [{v[is1].min():.3f}, {v[is1].max():.3f}]   FY2 [{v[~is1].min():.3f}, {v[~is1].max():.3f}]")
    assert v[is1].min() >= -1e-9 and v[is1].max() <= 1 + 1e-9

# (d) nwh_dist = 0 exactly when nwh = 0
assert ((ent["nwh_dist_raw"].to_numpy() > 0) == (ent["nwh"].to_numpy() == 1)).all()

# (e) promptly bins agree with lag_days (1 = 0-9, 2 = 10-29, 3 = 30+ days)
bins = pd.cut(ent["lag_days_raw"], [-np.inf, 9, 29, np.inf], labels=[1, 2, 3]).astype(int)
print("promptly vs lag bins, mismatches:", int((bins.to_numpy() != ent["promptly"].to_numpy()).sum()))

# (f) weekend flag agrees with the weekday of entered_date (Sat = 5, Sun = 6)
wk = np.select([ent["wday"] == 5, ent["wday"] == 6], [1, 2], 0)
print("weekend vs weekday, mismatches:", int((wk != ent["weekend"].to_numpy()).sum()))

## 6. Save

In [ ]:
fe.save_encoding(X, feature_info, params,
                 DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")
print("saved:", DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")

## 7. How later notebooks use it

Load once, then choose the feature groups per run. Write `fe.groups_label(groups)` into the `feature_groups` column of `results/runs.csv`.

In [ ]:
X, feature_info, params = fe.load_encoding(DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")

main = fe.FEATURE_GROUPS_MAIN
X_main, names_main = fe.select_columns(X, feature_info, main)
print(fe.groups_label(main), X_main.shape)

all_on = {g: True for g in fe.ALL_GROUPS}
X_all, _ = fe.select_columns(X, feature_info, all_on)
print(fe.groups_label(all_on), X_all.shape)

proposal_variant = {**main, "calendar_flags": False, "lag_days": False}
print(fe.groups_label(proposal_variant), fe.select_columns(X, feature_info, proposal_variant)[0].shape)

## Notes for the thesis (Ch. 3, feature engineering)

- Fill in the numbers from sections 2, 4 and 5 (entries with changed line order, the single 4-line entry, mismatch counts).
- `user` and `tax_rate` are constant inside every entry, so they are encoded once per entry, not per line slot.
- One entry has 4 lines; it alone uses slot 4. Check later where it ranks in the one-hot runs: if it appears at the top, it is a false positive caused by the encoding (the entry is normal), worth one sentence.
- Account / user / tax categories are taken from the whole ledger (no labels involved); all numeric scalers are fitted on FY1 only (D10).